In [6]:
"""
Cheap vs. clean - data loader
Pulls hourly DK1/DK2 day-ahead prices, wind/solar production and CO2
intensity of consumed electricity from Energinet's Energi Data Service
(free, no API key), and saves one tidy file: data/merged.csv

Study period: calendar year 2024 (fully hourly, before the
switch to 15-minute day-ahead prices in Oct 2025).

Usage:
    pip install pandas requests
    python get_data.py

Carbon intensity: Energinet's CO2Emis dataset (5-minute, g/kWh,
consumption-based, averaged here to hourly). If you later get
Electricity Maps CSVs, drop them in data/em/ and they are used instead.
"""

import glob
import json
from pathlib import Path

import pandas as pd
import requests

START = "2024-01-01T00:00"
END = "2025-01-01T00:00"          # exclusive
AREAS = ["DK1", "DK2"]
BASE = "https://api.energidataservice.dk/dataset/"
DATA = Path("data")
EM_DIR = DATA / "em"              # optional Electricity Maps CSVs


# ---------------------------------------------------------------------
# 1. Energi Data Service (no API key needed)
# ---------------------------------------------------------------------
def fetch_eds(dataset: str, time_col: str = "HourUTC") -> pd.DataFrame:
    """Download one dataset for both price areas, cache it as CSV."""
    # Period in the file name, so changing START/END never reuses old data
    cache = DATA / f"{dataset}_{START[:10]}_{END[:10]}.csv"
    if cache.exists():
        return pd.read_csv(cache, parse_dates=[time_col])

    params = {
        "start": START,
        "end": END,
        "filter": json.dumps({"PriceArea": AREAS}),
        "timezone": "UTC",        # avoid daylight-saving headaches
        "limit": 0,               # 0 = all rows
    }
    print(f"Downloading {dataset} ...")
    r = requests.get(BASE + dataset, params=params, timeout=300)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["records"])
    df[time_col] = pd.to_datetime(df[time_col])
    DATA.mkdir(exist_ok=True)
    df.to_csv(cache, index=False)
    print(f"{dataset}: {len(df):,} rows")
    return df


def get_prices() -> pd.DataFrame:
    df = fetch_eds("Elspotprices")
    return df[["HourUTC", "PriceArea", "SpotPriceEUR"]].rename(
        columns={"PriceArea": "area", "SpotPriceEUR": "price_eur_mwh"}
    )


def get_production() -> pd.DataFrame:
    df = fetch_eds("ProductionConsumptionSettlement")
    # Sum every wind / solar column, whatever their exact names are
    wind_cols = [c for c in df.columns if "Wind" in c]
    solar_cols = [c for c in df.columns if "Solar" in c]
    out = df[["HourUTC", "PriceArea"]].rename(columns={"PriceArea": "area"})
    out["wind_mwh"] = df[wind_cols].fillna(0).sum(axis=1)
    out["solar_mwh"] = df[solar_cols].fillna(0).sum(axis=1)
    if "GrossConsumptionMWh" in df.columns:
        out["consumption_mwh"] = df["GrossConsumptionMWh"]
        out["wind_share"] = out["wind_mwh"] / out["consumption_mwh"]
    return out


def get_carbon_energinet() -> pd.DataFrame:
    """5-minute CO2 intensity -> hourly mean (g CO2 per kWh consumed)."""
    df = fetch_eds("CO2Emis", time_col="Minutes5UTC")
    df["HourUTC"] = pd.to_datetime(df["Minutes5UTC"]).dt.floor("h")
    out = (df.groupby(["HourUTC", "PriceArea"], as_index=False)["CO2Emission"]
             .mean()
             .rename(columns={"PriceArea": "area",
                              "CO2Emission": "carbon_g_kwh"}))
    return out


# ---------------------------------------------------------------------
# 2. Optional: Electricity Maps CSVs (if you ever get them)
# ---------------------------------------------------------------------
def find_col(df: pd.DataFrame, *keywords: str) -> str:
    """Find a column whose name contains all keywords (case-insensitive)."""
    for c in df.columns:
        if all(k.lower() in c.lower() for k in keywords):
            return c
    raise KeyError(f"No column with {keywords}. Columns: {list(df.columns)}")


def get_carbon_em(files) -> pd.DataFrame:
    frames = []
    for f in files:
        df = pd.read_csv(f)
        t = find_col(df, "datetime")
        z = find_col(df, "zone")
        ci = find_col(df, "carbon intensity", "direct")
        frames.append(pd.DataFrame({
            "HourUTC": pd.to_datetime(df[t], utc=True).dt.tz_localize(None),
            "area": df[z].str.replace("DK-", "", regex=False),  # DK-DK1 -> DK1
            "carbon_g_kwh": df[ci],
        }))
    out = pd.concat(frames).drop_duplicates(["HourUTC", "area"])
    return out[(out.HourUTC >= START) & (out.HourUTC < END)]


def get_carbon() -> pd.DataFrame:
    files = sorted(glob.glob(str(EM_DIR / "*.csv")))
    if files:
        print("Carbon intensity: Electricity Maps CSVs")
        return get_carbon_em(files)
    print("Carbon intensity: Energinet CO2Emis")
    return get_carbon_energinet()


# ---------------------------------------------------------------------
# 3. Merge + sanity checks
# ---------------------------------------------------------------------
def main():
    prices = get_prices()
    prod = get_production()
    carbon = get_carbon()

    for d in (prices, prod, carbon):
        d["HourUTC"] = pd.to_datetime(d["HourUTC"]).dt.tz_localize(None)

    df = (prices.merge(carbon, on=["HourUTC", "area"], how="inner")
                .merge(prod, on=["HourUTC", "area"], how="left")
                .sort_values(["area", "HourUTC"]))
    df["HourDK"] = (df["HourUTC"].dt.tz_localize("UTC")
                    .dt.tz_convert("Europe/Copenhagen"))

    print("\nRows per area (expect ~8,784 - 2024 is a leap year):")
    print(df.groupby("area").size())
    print("\nMissing values:")
    print(df.isna().sum()[lambda s: s > 0])
    print("\nQuick look - price vs carbon correlation:")
    print(df.groupby("area")[["price_eur_mwh", "carbon_g_kwh"]]
            .corr().iloc[0::2, -1])

    df.to_csv(DATA / "merged.csv", index=False)
    print(f"\nSaved {DATA / 'merged.csv'}")


if __name__ == "__main__":
    main()

Elspotprices: 17,568 rows
ProductionConsumptionSettlement: 17,568 rows
Carbon intensity: Energinet CO2Emis
CO2Emis: 210,792 rows

Rows per area (expect ~8,784 - 2024 is a leap year):
area
DK1    8783
DK2    8783
dtype: int64

Missing values:
Series([], dtype: int64)

Quick look - price vs carbon correlation:
area               
DK1   price_eur_mwh    0.255297
DK2   price_eur_mwh    0.042930
Name: carbon_g_kwh, dtype: float64

Saved data/merged.csv
